In [1]:


import feedparser

# L'URL correcte pointe vers le flux RSS (XML) de l'ANSSI
url = "https://www.cert.ssi.gouv.fr/avis/feed/"
rss_feed = feedparser.parse(url)

# Parcourir chaque entrée du flux
for entry in rss_feed.entries:
    print("Titre :", entry.title)
    print("Description :", entry.description)
    print("Lien :", entry.link)
    print("Date :", entry.published)
    
    # Récupération de l'ID unique si disponible
    if 'id' in entry:
        print("ID :", entry.id)
        
    print("-" * 60)

Titre : Multiples vulnérabilités dans Google Chrome (05 juin 2026)
Description : De multiples vulnérabilités ont été découvertes dans Google Chrome. Elles permettent à un attaquant de provoquer un problème de sécurité non spécifié par l'éditeur.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0692/
Date : Fri, 05 Jun 2026 00:00:00 +0000
ID : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0692/
------------------------------------------------------------
Titre : Multiples vulnérabilités dans Microsoft Azure Linux (05 juin 2026)
Description : De multiples vulnérabilités ont été découvertes dans Microsoft Azure Linux. Elles permettent à un attaquant de provoquer un problème de sécurité non spécifié par l'éditeur.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0693/
Date : Fri, 05 Jun 2026 00:00:00 +0000
ID : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0693/
------------------------------------------------------------
Titre : Multiples vulnérabilités dans le noy

Pour voir tout ce qu'il y a comme flux scrapable

In [2]:
import requests
import re
url = "https://www.cert.ssi.gouv.fr/alerte/CERTFR-2024-ALE-001/json/"
response = requests.get(url)
data = response.json()
#Extraction des CVE reference dans la clé cves du dict data
ref_cves=list(data["cves"])
#attention il s’agit d’une liste des dictionnaires avec name et url comme clés
print( "CVE référencés ", ref_cves)
# Extraction des CVE avec une regex
cve_pattern = r"CVE-\d{4}-\d{4,7}"
cve_list = list(set(re.findall(cve_pattern, str(data))))
print("CVE trouvés :", cve_list)


CVE référencés  [{'name': 'CVE-2023-46805', 'url': 'https://www.cve.org/CVERecord?id=CVE-2023-46805'}, {'name': 'CVE-2024-21887', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-21887'}, {'name': 'CVE-2024-21893', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-21893'}, {'name': 'CVE-2024-21888', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-21888'}, {'name': 'CVE-2024-22024', 'url': 'https://www.cve.org/CVERecord?id=CVE-2024-22024'}]
CVE trouvés : ['CVE-2024-22024', 'CVE-2023-46805', 'CVE-2024-21887', 'CVE-2024-21888', 'CVE-2024-21893']


Exemple de connexion à l'API CVE :

In [3]:
import requests
cve_id = "CVE-2023-24488"
url = f"https://cveawg.mitre.org/api/cve/{cve_id}"
response = requests.get(url)
data = response.json()
# Extraire la description
description = data["containers"]["cna"]["descriptions"][0]["value"]
# Extraire le score CVSS
#ATTENTION tous les CVE ne contiennent pas nécessairement ce champ, gérez l’exception,
#ou peut etre au lieu de cvssV3_0 c’est cvssV3_1 ou autre clé
cvss_score =data["containers"]["cna"]["metrics"][0]["cvssV3_1"]["baseScore"]
cwe = "Non disponible"
cwe_desc="Non disponible"
problemtype = data["containers"]["cna"].get("problemTypes", {})
if problemtype and "descriptions" in problemtype[0]:
    cwe = problemtype[0]["descriptions"][0].get("cweId", "Non disponible")
    cwe_desc=problemtype[0]["descriptions"][0].get("description", "Non disponible")
# Extraire les produits affectés
affected = data["containers"]["cna"]["affected"]
for product in affected:
    vendor = product["vendor"]
    product_name = product["product"]
    versions = [v["version"] for v in product["versions"] if v["status"] == "affected"]
    print(f"Éditeur : {vendor}, Produit : {product_name}, Versions : {', '.join(versions)}")
# Afficher les résultats
print(f"CVE : {cve_id}")
print(f"Description : {description}")
print(f"Score CVSS : {cvss_score}")
print(f"Type CWE : {cwe}")
print(f"CWE Description : {cwe_desc}")


Éditeur : Citrix, Produit : Citrix ADC and Citrix Gateway , Versions : 13.1, 13.0, 12.1, 12.1-FIPS , 13.1-FIPS , 12.1-NDcPP
CVE : CVE-2023-24488
Description : Cross site scripting vulnerability in Citrix ADC and Citrix Gateway  in allows and attacker to perform cross site scripting
Score CVSS : 6.1
Type CWE : CWE-79
CWE Description : CWE-79 Improper Neutralization of Input During Web Page Generation ('Cross-site Scripting')


Exemple de connexion à l’API EPSS:

In [4]:
import requests
# URL de l'API EPSS pour récupérer la probabilité d'exploitation
cve_id = "CVE-2023-46805"
url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
# Requête GET pour récupérer les données JSON
response = requests.get(url)
data = response.json()
# Extraire le score EPSS
epss_data = data.get("data", [])
if epss_data:
    epss_score = epss_data[0]["epss"]
    print(f"CVE : {cve_id}")
    print(f"Score EPSS : {epss_score}")
else:
    print(f"Aucun score EPSS trouvé pour {cve_id}")


CVE : CVE-2023-46805
Score EPSS : 0.943670000


In [ ]:
import feedparser
import requests
import re
import pandas as pd
import time

# --- PARAMÈTRES DE LIMITATION ---
MAX_BULLETINS = 100         
MAX_CVE_PER_BULLETIN = 100

# --- FONCTIONS UTILITAIRES ---
def get_base_severity(score):
    """Détermine la gravité Base Severity selon le score CVSS."""
    try:
        score = float(score)
        if score == 0.0: return "None"
        elif 0.1 <= score <= 3.9: return "Low"
        elif 4.0 <= score <= 6.9: return "Medium"
        elif 7.0 <= score <= 8.9: return "High"
        elif 9.0 <= score <= 10.0: return "Critical"
    except (ValueError, TypeError):
        return "Non disponible"

def fetch_epss_score(cve_id):
    """Récupère la probabilité d'exploitation via l'API FIRST (EPSS)."""
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if data.get("data"):
            return data["data"][0]["epss"]
    except Exception:
        pass
    return "Non disponible"

def fetch_cve_details(cve_id):
    """Récupère et structure les détails d'un CVE depuis l'API de MITRE."""
    url = f"https://cveawg.mitre.org/api/cve/{cve_id}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return None
        data = response.json()
    except Exception:
        return None

    cna_container = data.get("containers", {}).get("cna", {})
    if not cna_container: return None

    # Extraction et NETTOYAGE de la description
    # C'est cette ligne qui sauve ton fichier CSV en supprimant les sauts de ligne
    descriptions = cna_container.get("descriptions", [])
    description = descriptions[0].get("value", "Non disponible") if descriptions else "Non disponible"
    description = description.replace('\n', ' ').replace('\r', '') 

    cvss_score = "Non disponible"
    metrics_list = cna_container.get("metrics", [])
    if metrics_list:
        for metric in metrics_list:
            for version_key in ["cvssV3_1", "cvssV3_0", "cvssV2_0"]:
                if version_key in metric:
                    cvss_score = metric[version_key].get("baseScore", cvss_score)
                    break
            if cvss_score != "Non disponible": break

    cwe = "Non disponible"
    problem_types = cna_container.get("problemTypes", [])
    if problem_types and "descriptions" in problem_types[0]:
        cwe_descriptions = problem_types[0]["descriptions"]
        if cwe_descriptions:
            cwe = cwe_descriptions[0].get("cweId", "Non disponible")

    vendor, product_name, versions_str = "Inconnu", "Inconnu", "Inconnues"
    products_list = cna_container.get("affected", [])
    if products_list:
        first_prod = products_list[0]
        vendor = first_prod.get("vendor", "Inconnu")
        product_name = first_prod.get("product", "Inconnu")
        versions = [v.get("version", "Inconnue") for v in first_prod.get("versions", []) if v.get("status") == "affected"]
        versions_str = ", ".join(versions) if versions else "Non spécifiées"

    return {
        "description": description,
        "cvss_score": cvss_score,
        "cwe": cwe,
        "vendor": vendor,
        "product": product_name,
        "versions": versions_str
    }

# --- PIPELINE PRINCIPAL : ÉTAPE 4 ---
def consolidation_pipeline():
    print("Initialisation de l'extraction des données ANSSI...")
    rss_url = "https://www.cert.ssi.gouv.fr/avis/feed/"
    rss_feed = feedparser.parse(rss_url)
    
    lignes_dataframe = []
    
    bulletins_a_traiter = rss_feed.entries[:MAX_BULLETINS]
    print(f"Traitement limité à {MAX_BULLETINS} bulletin(s) sur {len(rss_feed.entries)} disponibles.\n")
    
    for entry in bulletins_a_traiter:
        titre_anssi = entry.title
        lien_anssi = entry.link
        
        # Formatage propre de la date
        if hasattr(entry, 'published_parsed') and entry.published_parsed:
            date_pub = time.strftime("%Y-%m-%d", entry.published_parsed)
        else:
            date_pub = entry.published.replace(",", "") 
            
        id_anssi_match = re.search(r'(CERTFR-\d{4}-(AVI|ALE)-\d+)', lien_anssi)
        id_anssi = id_anssi_match.group(1) if id_anssi_match else "Inconnu"
        type_bulletin = "Avis" if "-AVI-" in id_anssi else "Alerte" if "-ALE-" in id_anssi else "Inconnu"
        
        print(f"[+] Bulletin : {id_anssi}")
        
        json_url = lien_anssi.rstrip('/') + "/json/"
        try:
            resp = requests.get(json_url, timeout=10)
            if resp.status_code == 200:
                anssi_data = resp.json()
                cves_trouves = list(set(re.findall(r"CVE-\d{4}-\d{4,7}", str(anssi_data))))
            else:
                cves_trouves = []
        except Exception:
            cves_trouves = []
            
        print(f"    -> {len(cves_trouves)} CVE(s) identifié(s).")
        
        cves_a_traiter = cves_trouves[:MAX_CVE_PER_BULLETIN]
        if len(cves_trouves) > MAX_CVE_PER_BULLETIN:
            print(f"    -> Limitation : analyse restreinte aux {MAX_CVE_PER_BULLETIN} premiers CVE.")
        
        for cve in cves_a_traiter:
            print(f"       * Enrichissement {cve}...", end=" ")
            
            mitre_data = fetch_cve_details(cve)
            epss_score = fetch_epss_score(cve)
            
            time.sleep(1) 
            
            if mitre_data:
                base_severity = get_base_severity(mitre_data["cvss_score"])
                
                lignes_dataframe.append({
                    "ID ANSSI": id_anssi,
                    "Titre ANSSI": titre_anssi,
                    "Type": type_bulletin,
                    "Date": date_pub,
                    "CVE": cve,
                    "CVSS": mitre_data["cvss_score"],
                    "Base Severity": base_severity,
                    "CWE": mitre_data["cwe"],
                    "EPSS": epss_score,
                    "Lien": lien_anssi,
                    "Description": mitre_data["description"],
                    "Éditeur": mitre_data["vendor"],
                    "Produit": mitre_data["product"],
                    "Versions affectées": mitre_data["versions"]
                })
                print("Ok.")
            else:
                print("Échec (données Mitre indisponibles).")

    print("\n=== Consolidation Terminée ===")
    df = pd.DataFrame(lignes_dataframe)
    return df

# --- EXÉCUTION ---
if __name__ == "__main__":
    df_final = consolidation_pipeline()
    df_final.to_csv("donnees_consolidees_anssi.csv", index=False, sep=";")
    print("\nFichier CSV généré avec succès. Tu peux l'ouvrir dans VS Code sans erreurs de formatage.")

Initialisation de l'extraction des données ANSSI...
Traitement limité à 100 bulletin(s) sur 40 disponibles.

[+] Bulletin : CERTFR-2026-AVI-0692
    -> 429 CVE(s) identifié(s).
    -> Limitation : analyse restreinte aux 100 premiers CVE.
       * Enrichissement CVE-2026-11151... Ok.
       * Enrichissement CVE-2026-10892... Ok.
       * Enrichissement CVE-2026-10885... Ok.
       * Enrichissement CVE-2026-11302... Ok.
       * Enrichissement CVE-2026-11091... Ok.
       * Enrichissement CVE-2026-11229... Ok.
       * Enrichissement CVE-2026-11126... Ok.
       * Enrichissement CVE-2026-11069... Ok.
       * Enrichissement CVE-2026-11142... Ok.
       * Enrichissement CVE-2026-10897... Ok.
       * Enrichissement CVE-2026-11228... Ok.
       * Enrichissement CVE-2026-10898... Ok.
       * Enrichissement CVE-2026-11081... Ok.
       * Enrichissement CVE-2026-10900... 